In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('')))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.graph_objects as go
import plotly.subplots as sp
from datetime import datetime

# LUCiD imports
from tools.geometry import generate_detector
from tools.simulation import setup_event_simulator
from tools.generate import read_photon_data_from_photonsim
from tools.utils import spherical_to_cartesian, base_dir_path
from tools.optimization.optimize import get_detector_bounds
from tools.utils import generate_random_event_params

## Configuration Parameters

Adjust these parameters to customize the visualization:

In [ ]:
# Configuration parameters
CONFIG = {
    'detector_config': base_dir_path() + 'config/SK_geom_config.json',  # Detector configuration file
    'data_file': '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root',
    'detector_type': 'Cylinder',  # Detector geometry type
    'entry_idx': 2,  # Which entry to use from ROOT file
    'n_photons': 3_00_000,  # Number of photons to simulate
    'K': 6,  # Number of scattering iterations
    'seed': 71900,  # Random seed
    'min_charge': 1.0,  # Minimum charge threshold for display
    'color_by': 'charge',  # Color sensor hits by 'charge' or 'time'
    'dark_theme': False,  # Use dark theme for disc visualizations
    'log_scale': False,  # Use log scale for disc visualizations
    'save_figures': True  # Save figures to files
}

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Setup Detector and Simulators

In [ ]:
# Setup detector
print("Setting up detector...")
detector = generate_detector(CONFIG['detector_config'])
sensor_positions = jnp.array(detector.all_points)
detector_bounds = get_detector_bounds(detector)
n_sensors = len(sensor_positions)

print(f"  Type: {CONFIG['detector_type']}")
print(f"  Sensors: {n_sensors:,}")
print(f"  Bounds: {detector_bounds}")

# Sensor parameters
sensor_params = (
    jnp.array(50.0),    # scatter_length
    jnp.array(0.1),     # reflection_rate
    jnp.array(100.0),   # absorption_length
    jnp.array(0.001)    # gumbel_softmax_temperature
)

In [ ]:
# Setup simulators
print("Setting up simulators...")

# Prediction simulator (regular physics simulation)
prediction_simulator = setup_event_simulator(
    json_filename=CONFIG['detector_config'],
    max_sensors_per_cell=4,
    n_photons=CONFIG['n_photons'],
    temperature=0.0,
    K=CONFIG['K'],
    detector_type=CONFIG['detector_type'],
    is_data=False
)

# Data simulator (transforms reference photons)
data_simulator = setup_event_simulator(
    json_filename=CONFIG['detector_config'],
    max_sensors_per_cell=4,
    n_photons=CONFIG['n_photons'],
    temperature=0.0,  # Zero temperature for data mode
    K=CONFIG['K'],
    detector_type=CONFIG['detector_type'],
    is_data=True
)

print("  Simulators ready")

## Load Reference Photons and Generate Track Parameters

In [ ]:
# Load photon data from ROOT file
print(f"Loading reference photons from ROOT file...")
photon_data = read_photon_data_from_photonsim(CONFIG['data_file'], CONFIG['entry_idx'])
photon_data['N'] = len(photon_data['photon_origins'])

print(f"  Number of photons: {photon_data['N']:,}")
print(f"  Primary energy: {photon_data['energy']:.1f} MeV")

# Generate track parameters
print("\nGenerating track parameters...")
key = jax.random.PRNGKey(CONFIG['seed'])
track_position, track_direction, _ = generate_random_event_params(key, detector_bounds)
track_energy = photon_data['energy']

track_position = jnp.array([-10., 0., 0.])
track_direction = jnp.array([1., 0., 0.])


original_direction = jnp.array([0.0, 0.0, 1.0])
true_direction_norm = track_direction / (jnp.linalg.norm(track_direction) + 1e-8)

# Rotation axis = cross product of original and target directions
rotation_axis = jnp.cross(original_direction, true_direction_norm)
axis_norm = jnp.linalg.norm(rotation_axis)

# Handle case where directions are parallel (axis_norm ~ 0)
rotation_axis = jnp.where(
    axis_norm < 1e-6,
    jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
    rotation_axis / (axis_norm + 1e-8)
)

# Rotation angle = arccos of dot product
rotation_angle = jnp.arccos(jnp.clip(
    jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
))

# Set rotation parameters
photon_data['rotation_axis'] = rotation_axis
photon_data['rotation_angle'] = rotation_angle
photon_data['apply_rotation'] = jnp.array(True)

# Set translation parameters to move from origin to true_position
photon_data['apply_translation'] = jnp.array(True)
photon_data['translation_vector'] = track_position


print(f"  Position: [{track_position[0]:.3f}, {track_position[1]:.3f}, {track_position[2]:.3f}] m")
print(f"  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]")
print(f"  Energy: {track_energy:.1f} MeV")

## Simulate Events

In [ ]:
# Simulate events
print("Simulating events...")
event_key = jax.random.PRNGKey(CONFIG['seed'] + 1000)

# Prediction-like event
print("  Generating prediction-like event...")
# Convert direction for prediction simulator
theta = jnp.arccos(jnp.clip(track_direction[2], -1.0, 1.0))
phi = jnp.arctan2(track_direction[1], track_direction[0])
direction_angles = jnp.array([theta, phi])

prediction_params = (track_energy, track_position, direction_angles)
prediction_charges, prediction_times = prediction_simulator(prediction_params, sensor_params, event_key)

# Data-like event
print("  Generating data-like event...")
data_params = (track_energy, track_position, track_direction)
data_charges, data_times = data_simulator(data_params, sensor_params, event_key, photon_data)

print("  Events generated successfully")

## Event Analysis and Statistics

In [ ]:
# Create statistical comparison plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 6))

# Filter active sensors
pred_active = prediction_charges > CONFIG['min_charge']
data_active = data_charges > CONFIG['min_charge']

pred_charges_active = prediction_charges[data_active & pred_active]
pred_times_active = prediction_times[data_active & pred_active]
data_charges_active = data_charges[data_active]
data_times_active = data_times[data_active]

# Charge distributions
ax1.hist(pred_charges_active, bins=150, range=[0,20], alpha=0.7, label='Prediction-like', color='blue', density=True)
ax1.hist(data_charges_active, bins=150, range=[0,20], alpha=0.7, label='Data-like', color='red', density=True)
ax1.set_xlabel('Charge')
ax1.set_ylabel('Density')
ax1.set_title('Charge Distribution Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Time distributions
ax2.hist(pred_times_active, bins=100, range=[0,200], alpha=0.7, label='Prediction-like', color='blue', density=False)
ax2.hist(data_times_active, bins=100, range=[0,200], alpha=0.7, label='Data-like', color='red', density=False)
ax2.set_xlabel('Time [ns]')
ax2.set_ylabel('Density')
ax2.set_title('Time Distribution Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Charge vs Time scatter
ax3.scatter(pred_charges_active, pred_times_active, alpha=0.6, s=10, label='Prediction-like', color='blue')
ax3.scatter(data_charges_active, data_times_active, alpha=0.6, s=10, label='Data-like', color='red')
ax3.set_xlabel('Charge')
ax3.set_ylabel('Time [ns]')
ax3.set_title('Charge vs Time Correlation')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Summary statistics
stats_text = f"""
Prediction-like Event:
  Active sensors: {len(pred_charges_active):,}
  Mean charge: {np.mean(pred_charges_active):.2f} ± {np.std(pred_charges_active):.2f}
  Mean time: {np.mean(pred_times_active):.1f} ± {np.std(pred_times_active):.1f} ns

Data-like Event:
  Active sensors: {len(data_charges_active):,}
  Mean charge: {np.mean(data_charges_active):.2f} ± {np.std(data_charges_active):.2f}
  Mean time: {np.mean(data_times_active):.1f} ± {np.std(data_times_active):.1f} ns

Track Parameters:
  Energy: {track_energy:.1f} MeV
  Position: [{track_position[0]:.2f}, {track_position[1]:.2f}, {track_position[2]:.2f}] m
  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]
"""

ax4.hist2d(prediction_times[data_active & pred_active], data_times[data_active & pred_active], bins=(np.linspace(0,300,40), np.linspace(0,300,40)))
ax4.plot(range(0,300), range(0,300))
ax4.set_xlim(0,300)
ax4.set_ylim(0,300)

plt.tight_layout()

if CONFIG['save_figures']:
    figures_dir = Path(base_dir_path()) / 'figures'
    detector_name = Path(CONFIG['detector_config']).stem.replace('_geom_config', '')
    filename = figures_dir / f'{detector_name}_statistical_comparison.png'
    _ = plt.savefig(str(filename), dpi=300, bbox_inches='tight')
    print(f"Saved: {filename}")